# English-to-Urdu Translation using Opus-MT

Pre-trained Helsinki-NLP Opus-MT model (no fine-tuning needed - ready to use!)

**Model:** Helsinki-NLP/Opus-MT-en-ur (pre-trained translation model)

**Advantage:** Already trained on translation tasks - produces clean Urdu output immediately

## 1. Install and Load Model

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

print(f"CUDA available: {torch.cuda.is_available()}")

# Load pre-trained Opus-MT model (already fine-tuned for English-Urdu)
model_name = "Helsinki-NLP/Opus-MT-en-ur"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
model = model.cuda() if torch.cuda.is_available() else model

print(f"✓ Model loaded: {model_name}")
print(f"  This model is already fine-tuned for English→Urdu translation")

## 2. Translation Function

In [ ]:
def translate(text, model, tokenizer):
    """Translate English to Urdu using Opus-MT."""
    inputs = tokenizer(text, return_tensors="pt", max_length=512, truncation=True).to(model.device)
    
    # Generate translation
    generated = model.generate(
        **inputs,
        max_length=512,
        num_beams=4,
        early_stopping=True,
        no_repeat_ngram_size=2,
    )
    
    return tokenizer.decode(generated[0], skip_special_tokens=True)

print("✓ Translation function ready")

## 3. Test Translations

In [ ]:
test_sentences = [
    "how are you",
    "what is your name",
    "i love my country",
    "the weather is beautiful today",
    "where is the school",
    "he is a good person",
    "please help me",
    "i am going home",
    "tell zain i said hi",
    "thanks for the help",
    "i like the dog",
    "the thief ran away",
    "zain put out the fire",
]

print("\n" + "="*70)
print("Opus-MT English-to-Urdu Translations (Pre-trained)")
print("="*70)
for sent in test_sentences:
    translation = translate(sent, model, tokenizer)
    print(f"EN: {sent}")
    print(f"UR: {translation}")
    print("-" * 70)

## 4. Batch Translation (for large datasets)

In [ ]:
def batch_translate(texts, model, tokenizer, batch_size=8):
    """Translate multiple sentences efficiently."""
    translations = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        inputs = tokenizer(batch, return_tensors="pt", padding=True, truncation=True, max_length=512).to(model.device)
        generated = model.generate(**inputs, max_length=512, num_beams=4)
        batch_translations = tokenizer.batch_decode(generated, skip_special_tokens=True)
        translations.extend(batch_translations)
    return translations

print("✓ Batch translation function ready")

## 5. Example: Translate Your Dataset

In [ ]:
# Example: Load and translate your English corpus
import os

if os.path.exists("Dataset/english-corpus.txt"):
    with open("Dataset/english-corpus.txt", "r", encoding="utf-8") as f:
        en_sentences = [line.strip() for line in f.readlines()][:20]  # First 20 sentences
    
    print(f"\nTranslating first 20 sentences from Dataset...\n")
    print("="*70)
    
    ur_translations = batch_translate(en_sentences, model, tokenizer)
    
    for en, ur in zip(en_sentences, ur_translations):
        print(f"EN: {en}")
        print(f"UR: {ur}")
        print("-" * 70)
else:
    print("Dataset not found. Using test sentences above.")